In [0]:
%pip install dbt-databricks

In [0]:
%sh ls /Workspace/Users/ashish@alphainterbiz.com/atliq-commerce-capstone/phase1-batch/dbt/atliq_gold

In [0]:
import os

# Read the token from the Key Vault-backed secret scope
token = dbutils.secrets.get(scope="atliq", key="dbt-token")

dbt_project_dir = "/Workspace/Users/ashish@alphainterbiz.com/atliq-commerce-capstone/phase1-batch/dbt/atliq_gold"

# Write a profiles.yml into the project dir (uses the token directly, from the secret)
profiles_yaml = f"""
atliq_gold:
  target: dev
  outputs:
    dev:
      type: databricks
      host: adb-7405614391071920.0.azuredatabricks.net
      http_path: /sql/1.0/warehouses/28d7b90aaccdb4c8
      token: {token}
      catalog: atliq
      schema: gold
      threads: 4
"""

with open(f"{dbt_project_dir}/profiles.yml", "w") as f:
    f.write(profiles_yaml)

print("profiles.yml written to project dir")
print("token length:", len(token))

In [0]:
import subprocess

# install packages first (dbt_utils, needed for the accepted_range test)
deps = subprocess.run(["dbt", "deps", "--project-dir", dbt_project_dir], capture_output=True, text=True)
print(deps.stdout, deps.stderr)

# then build models + run tests
result = subprocess.run(
    ["dbt", "build", "--project-dir", dbt_project_dir, "--profiles-dir", dbt_project_dir],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise Exception("dbt build failed")
print("dbt build succeeded")